# Shared incident scenarios

This notebook is the single source of synthetic incident evidence. Run all cells to export `data/scenarios.json` for **every** agent and `data/expected_results.json` for evaluation only. Never provide the expected answers to agents or their tools. A is critical, B is the normal high-load path, and C is an ambiguous startup failure. Metrics are a snapshot; log timestamps distinguish current from stale evidence.

In [1]:
SCENARIOS = [
    {
        'id': 'A',
        'label': 'API latency and DB pool exhaustion',
        'metrics': {'latency_p95_ms': 1400, 'error_rate_pct': 18.5, 'cpu_pct': 62,
                    'requests_per_min': 1800, 'db_connections_in_use': 100,
                    'db_pool_limit': 100, 'availability_pct': 81.5},
        'logs': [
            '2026-11-16T10:00:00Z ERROR api request timeout waiting for DB connection (pool 100/100)',
            '2026-11-16T10:00:05Z WARN db checkout queue length 73; wait 1200ms',
            '2026-11-16T10:00:10Z INFO deployment stable; no release in the incident window',
        ],
        'deployment_history': [
            {'version': '2.4.0', 'status': 'healthy', 'change': 'deployed before incident window'},
        ],
        'previous_version': {'version': '2.3.9', 'health': 'healthy before update',
                             'config': 'DB pool limit 100'},
    },
    {
        'id': 'B',
        'label': 'High traffic and CPU with healthy DB pool',
        'metrics': {'latency_p95_ms': 480, 'error_rate_pct': 0.3, 'cpu_pct': 92,
                    'requests_per_min': 5200, 'db_connections_in_use': 30,
                    'db_pool_limit': 100, 'availability_pct': 99.9},
        'logs': [
            '2026-11-16T10:00:00Z INFO api traffic 5200 requests/min after scheduled campaign',
            '2026-11-16T10:00:05Z INFO db checkout wait 4ms; pool 30/100',
            '2026-11-16T10:00:10Z WARN cpu 92 percent; autoscaling capacity approaching limit',
        ],
        'deployment_history': [
            {'version': '2.4.0', 'status': 'healthy', 'change': 'no release in the traffic window'},
        ],
        'previous_version': {'version': '2.3.9', 'health': 'healthy before update',
                             'config': 'same capacity policy'},
    },
    {
        'id': 'C',
         'label': 'Startup failure after configuration change',
        'metrics': {'latency_p95_ms': 0, 'error_rate_pct': 100.0, 'cpu_pct': 7,
                    'requests_per_min': 0, 'db_connections_in_use': 2,
                    'db_pool_limit': 100, 'availability_pct': 0.0},
        'logs': [
            '2026-11-16T09:10:00Z WARN db transient timeout on previous release (resolved)',
            '2026-11-16T10:00:00Z ERROR app startup failed: missing AUTH_ISSUER setting',
            '2026-11-16T10:00:03Z WARN health check unavailable; no live request logs',
        ],
        'deployment_history': [
            {'version': '2.4.1', 'status': 'failed startup',
             'change': 'AUTH_ISSUER configuration changed at 10:00Z'},
            {'version': '2.4.0', 'status': 'healthy', 'change': 'previous running version'},
        ],
        'previous_version': {'version': '2.4.0', 'health': 'healthy before config change',
                             'config': 'AUTH_ISSUER present'},
    },
]

EXPECTED_RESULTS = {
    'A': {'category': 'critical', 'likely_cause': 'DB connection pool exhaustion',
          'safe_action': 'Investigate leaked/long-running DB connections; limit load and restore pool capacity cautiously.',
          'verification': 'DB checkout queue and request error/latency rates recover.'},
    'B': {'category': 'normal high load', 'likely_cause': 'Traffic-driven CPU saturation, not DB exhaustion',
          'safe_action': 'Monitor or scale application capacity; do not roll back or change DB pool blindly.',
          'verification': 'CPU and latency stabilize while availability remains high.'},
    'C': {'category': 'edge/ambiguous failure', 'likely_cause': 'Missing startup setting after config change; stale DB warning is inconclusive',
          'safe_action': 'Verify current startup logs and configuration; restore last-known-good config/version if confirmed.',
          'verification': 'Startup and health checks pass on the restored configuration.'},
}


## Export the same data for every teammate

Run this cell from the repository root or the `notebooks` directory. Only `scenarios.json` is passed to agents; `expected_results.json` is for testing and the final comparison. JSON serialization is deterministic so changes to either export are visible in version control.

In [2]:
import json
from pathlib import Path

def export(root=None):
    if root is None:
        cwd = Path.cwd()
        root = next((candidate for candidate in (cwd, cwd.parent)
                     if (candidate / 'notebooks' / 'incident_scenarios.ipynb').is_file()), None)
        if root is None:
            raise FileNotFoundError('Run from the repository root or notebooks directory')
    output_dir = Path(root) / 'data'
    output_dir.mkdir(parents=True, exist_ok=True)
    for filename, records in (('scenarios.json', SCENARIOS),
                              ('expected_results.json', EXPECTED_RESULTS)):
        (output_dir / filename).write_text(
            json.dumps(records, indent=2, ensure_ascii=True) + '\n', encoding='utf-8'
        )
    return output_dir

export()


WindowsPath('c:/Users/muthyalv/.copilot/repos/copilot-worktrees/incd-assist/vishruth-muthyala-hpeprod-jubilant-carnival/data')